# Load Skeletons

### Imports

In [1]:
import json
import numpy as np
import os

from agentic_neuron_proofreader.utils import img_util, util
from agentic_neuron_proofreader.data_modules.datasets import BrainDataset

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

### Subroutines

In [2]:
def get_img_path(brain_id):
    path = "exaspim_image_prefixes.json"
    img_prefix = util.read_json(path)[int(brain_id)]
    return os.path.join(img_prefix, "0")

## Section 1: Load Data

### Initializations

In [3]:
# Dataset info
brain_id = "794495"
segmentation_id = "raw.unet_449_splits_and_merges_900000"

# Paths
gt_path = f"gs://allen-nd-goog/ground_truth_tracings/{brain_id}/voxel"
fragments_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/swcs"

img_path = get_img_path(brain_id)
segmentation_path =  f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/"

### Load Data

In [4]:
# Parameters
# anisotropy: voxel-to-physical scaling factors in microns/voxel for (x, y, z).
#   ExaSPIM samples xy more finely than z, so voxels are not cubic. Used to
#   convert SWC coordinates (microns) <-> image voxel indices when reading patches.
anisotropy = (0.748, 0.748, 1.0)

# min_cable_length: drop fragments whose total skeleton path length (in microns)
#   is below this threshold. Filters out short, noisy UNet fragments at load time.
min_cable_length = 1000

# node_spacing: target distance (in microns) between neighboring nodes after
#   resampling each skeleton branch. Lower = finer geometry but more memory.
node_spacing = 5

# Create dataset
dataset = BrainDataset(
    fragments_path,
    gt_path,
    img_path,
    anisotropy=anisotropy,
    min_cable_length=min_cable_length,
    node_spacing=node_spacing,
)

# Report summary
print(dataset.fragments_graph.summary(prefix="Fragments"))
print(dataset.gt_graph.summary(prefix="GroundTruth"))

Load Graphs: 100%|██████████| 19/19 [00:45<00:00,  2.41s/it]


Fragments Graph
# Connected Components: 10,172
# Nodes: 4,281,310
# Edges: 4,271,138
Memory Consumption: 6.62 GBs
GroundTruth Graph
# Connected Components: 19
# Nodes: 1,363,808
# Edges: 1,363,789
Memory Consumption: 6.67 GBs


### Save Dataset Cache

Pickles the two `SkeletonGraph` objects (the slow part to rebuild — ~15 min from GCS) plus the paths/parameters needed to re-open the lazy image readers. The `TensorStoreImage` itself is NOT pickled because it re-instantiates instantly.

Expected cache size is ~500–700 MB (mostly NetworkX adjacency + KDTree + a few NumPy arrays). This is much smaller than the "Memory Consumption" line in `summary()`, which reports total *process* memory (TensorStore caches, the SWC reader pool, etc.) rather than the graph's own footprint.

Use `load_skeletons_from_cache.ipynb` to reload without re-reading the SWCs.

In [5]:
cache_path = f"dataset_cache_{brain_id}.pkl"
dataset.save(cache_path)
print(f"Saved dataset cache: {cache_path} ({os.path.getsize(cache_path) / 1e9:.2f} GB)")

Saved dataset cache: dataset_cache_794495.pkl (0.59 GB)


### Visualization

In [6]:
segmentation = img_util.TensorStoreImage(segmentation_path)

I0523 20:48:53.569216 115008224 google_auth_provider.cc:149] Using credentials at zihan_gcs_token.json
I0523 20:48:53.570621 115008224 google_auth_provider.cc:165] Using ServiceAccount AuthProvider
